[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abhijeet1520/langgraph-workshop/blob/main/level-4-real-inbox/40_connect_your_inbox.ipynb)

# Connect Your Real Inbox

**Level 4** · Part of the [LangGraph Workshop](https://github.com/Abhijeet1520/langgraph-workshop).

**Provider setup:** set `OPENAI_API_KEY` / `OPENAI_API_BASE` / `OPENAI_MODEL` first — [docs/setup/providers.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/setup/providers.md). The first cells install requirements and verify the connection.

## Level 4 — Point the Agent at Your Real Inbox 📬

Level 3's agent ran on two hardcoded sample emails. Now we load **your real emails** into the same `EmailState` and run the identical graph — no rebuild.

**⚠️ Safety rules (read first):**
- **Read-only + draft-only in this workshop.** We fetch, classify, and *draft* replies. Nothing is ever sent, moved, or deleted.
- Use an app password + a **test label/folder**, not your main account if you can.
- LLM calls cost money & your emails contain PII — start with 5–10 emails.

**Steps:** (1) IMAP connect → (2) fetch a batch → (3) map to `EmailState` → (4) run the Level 3 graph → (5) review drafts.

## Install

In [ ]:
%pip install -q langgraph langchain langchain-core langchain-openai

## Provider + mailbox config

In [ ]:
# --- Workshop provider config (edit ONLY these 3 values) ---------------------
import os

os.environ.setdefault("OPENAI_API_KEY", "")   # <- paste your key
os.environ.setdefault("OPENAI_API_BASE", "")  # <- paste your base URL
os.environ.setdefault("OPENAI_MODEL", "openai/gpt-4.1-mini")  # <- your model id

# Verify
for _v in ("OPENAI_API_KEY", "OPENAI_API_BASE", "OPENAI_MODEL"):
    print(_v, "=", "set" if os.getenv(_v) else "**MISSING — see docs/setup/providers.md**")
# -----------------------------------------------------------------------------

# --- Mailbox config ----------------------------------------------------------
IMAP_HOST = "imap.gmail.com"        # Gmail; outlook: outlook.office365.com
IMAP_USER = "you@gmail.com"
IMAP_PASSWORD = ""                  # <- Gmail: 16-char app password (not your login)
FOLDER = "INBOX"                    # safer: a test label like "workshop-test"
MAX_EMAILS = 5                      # start small
# -----------------------------------------------------------------------------

In [ ]:
from langchain_openai import ChatOpenAI
import os

def make_llm(**overrides):
    """Provider-agnostic LLM factory used across the workshop."""
    cfg = dict(
        model=os.environ["OPENAI_MODEL"],
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.environ["OPENAI_API_BASE"],
        temperature=0,
    )
    cfg.update(overrides)
    return ChatOpenAI(**cfg)


llm = make_llm()
print("✅ LLM ready:", os.environ["OPENAI_MODEL"])


## 1. Connect (read-only)

Gmail app passwords: Google Account → Security → 2-Step Verification → App passwords. Requires 2FA. Outlook/others: any IMAP host works — just change `IMAP_HOST`.

In [ ]:
import imaplib

if not IMAP_PASSWORD:
    print("⚠️ Set IMAP_PASSWORD above first (Gmail: 16-char app password)")
else:
    mail = imaplib.IMAP4_SSL(IMAP_HOST)
    mail.login(IMAP_USER, IMAP_PASSWORD)
    mail.select(FOLDER, readonly=True)   # READ-ONLY
    print("✅ Connected:", IMAP_USER, "folder:", FOLDER)


## 2. Fetch a batch of emails

In [ ]:
import email
from email.header import decode_header

def fetch_emails(max_emails: int = 5) -> list[dict]:
    """Fetch latest emails as plain dicts: sender / subject / body."""
    typ, data = mail.search(None, "ALL")
    ids = data[0].split()[-max_emails:]
    out = []
    for eid in ids:
        typ, msg_data = mail.fetch(eid, "(RFC822)")
        msg = email.message_from_bytes(msg_data[0][1])

        def hdr(name: str) -> str:
            raw = msg.get(name, "")
            parts = decode_header(raw)
            return "".join(
                p.decode(enc or "utf-8", "replace") if isinstance(p, bytes) else p
                for p, enc in parts
            )

        body = ""
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    body = part.get_payload(decode=True).decode("replace", "ignore")
                    break
        elif msg.get_content_type() == "text/plain":
            body = msg.get_payload(decode=True).decode("replace", "ignore")

        out.append({"sender": hdr("From"), "subject": hdr("Subject"), "body": body[:4000]})
    return out

emails = fetch_emails(MAX_EMAILS)
print(f"✅ Fetched {len(emails)} emails")
for e in emails:
    print(" -", e["sender"][:40], "|", e["subject"][:50])


## 3. The Level 3 graph, unchanged

Copy of the EmailState graph from `30_email_agent_graph.ipynb` — same code, so you can see that **your inbox data just flows into it**.

In [ ]:
from typing import TypedDict, List, Dict, Any, Optional
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage

class EmailState(TypedDict):
    email: Dict[str, Any]
    is_spam: Optional[bool]
    draft_response: Optional[str]
    messages: List[Dict[str, Any]]

def read_email(state: EmailState):
    email = state["email"]
    print(f"Processing email from {email['sender']} — {email['subject']}")
    return {}

def classify_email(state: EmailState):
    email = state["email"]
    prompt = f"""You are an email triage assistant. Analyze this email and determine if it is spam.
Email:
From: {email['sender']}
Subject: {email['subject']}
Body: {email['body'][:1500]}
Answer with exactly SPAM or HAM (legitimate). Only return the answer."""
    response = llm.invoke([HumanMessage(content=prompt)])
    text = response.content.lower().strip()
    is_spam = "spam" in text and "ham" not in text
    return {"is_spam": is_spam}

def handle_spam(state: EmailState):
    print("  -> marked as spam (draft action only; nothing moved)")
    return {}

def drafting_response(state: EmailState):
    email = state["email"]
    prompt = f"""You are an email assistant. Draft a brief, professional reply to this email.
Email:
From: {email['sender']}
Subject: {email['subject']}
Body: {email['body'][:1500]}
Draft the reply only, ready for the human to review and send."""
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"draft_response": response.content}

email_graph = StateGraph(EmailState)
email_graph.add_node("read_email", read_email)
email_graph.add_node("classify_email", classify_email)
email_graph.add_node("handle_spam", handle_spam)
email_graph.add_node("drafting_response", drafting_response)

def route_email(state: EmailState) -> str:
    return "spam" if state["is_spam"] else "legitimate"

email_graph.add_edge(START, "read_email")
email_graph.add_edge("read_email", "classify_email")
email_graph.add_conditional_edges("classify_email", route_email, {
    "spam": "handle_spam",
    "legitimate": "drafting_response",
})
email_graph.add_edge("handle_spam", END)
email_graph.add_edge("drafting_response", END)

compiled = email_graph.compile()
print("✅ Graph compiled — identical structure to Level 3")


## 4. Run it on your inbox (drafts only)

In [ ]:
for e in emails:
    result = compiled.invoke({
        "email": e,
        "is_spam": None,
        "draft_response": None,
        "messages": [],
    })
    if result["is_spam"]:
        print(f"  [SPAM] {e['subject'][:50]}")
    else:
        print(f"  [DRAFT] {e['subject'][:50]}")
        print("   reply:", (result.get("draft_response") or "")[:200].replace("\n", " "))
        print()

# ⚠️ Nothing was sent, moved, or deleted — drafts printed to your screen only.


## Going further (carefully)

- **Auto-apply after trust:** only after reviewing many runs would you write results to a label/folder (IMAP `STORE`), and even then start with copying, not moving.
- **Reply sending (SMTP)** is possible but out of scope — always keep a human in the loop for anything that leaves your machine.
- **Cost/PII hygiene:** each email = 2 LLM calls (classify + draft). Batch small, run selectively (e.g. `UNSEEN` or a label filter), and never paste sensitive mail into a shared endpoint you don't trust.

Next: [Level 5 — what else can you automate?](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/levels/level-5-next-steps/what-next.md)